# Explorer API Client: Querying Endpoints

## Setup

Get your API key and secret from the Colab secrets.

Your API key and secret are available in your [Altmetric Explorer account](https://www.altmetric.com/explorer/settings). Click the key icon on the left hand side of the Colab window and add them as secrets named `API_KEY` and `API_SECRET` before running this step.

In [ ]:
from google.colab import userdata

API_KEY = userdata.get('API_KEY')
API_SECRET = userdata.get('API_SECRET')
API_URL = 'https://www.altmetric.com/explorer/api'

Install the API client from GitHub.

In [ ]:
!pip install https://github.com/altmetric/altmetric-explorer-api-client/tarball/main

Create a single client to reuse for every request below.

In [ ]:
from altmetric.explorer import api

api_client = api.Client(API_URL, API_KEY, API_SECRET)

## Choosing an endpoint

The client can call **any** Explorer API endpoint. You have two options:

* **Shorthand methods** for the most common endpoints: `get_attention_summary()`, `get_demographics()`, `get_journals()`, `get_mentions()`, `get_mention_sources()` and `get_research_outputs()`.
* **The generic `get(path, **filters)` method** for any endpoint at all, including endpoints added to the API *after* this client was released, which work without any client update.

This notebook shows the mechanics of both. The authoritative catalogue of endpoints, the data each returns and the filters each accepts lives in the [Explorer API documentation](https://docs.altmetric.com/explorer-api/). Refer to it rather than to a list baked into this client. For query construction, filter syntax, pagination and reading responses in depth, see the [Getting Started](https://colab.research.google.com/github/altmetric/altmetric-explorer-api-client/blob/main/docs/getting_started.ipynb) notebook.

### A shorthand method

Each shorthand simply calls a common path for you. Here is the attention summary, printing the `meta` block of the response:

In [ ]:
response = api_client.get_attention_summary()

for key, value in response.meta.items():
    print(f'{key}: {value}')

### Calling any endpoint with `get`

`get` takes the endpoint path followed by any filters as keyword arguments, builds the query string and signs the request for you. For example, mention sources of type `policy` or `blog` that were active in the last 3 days:

In [ ]:
response = api_client.get(
    'research_outputs/mention_sources',
    mention_sources_types=['type:policy', 'type:blog'],
    timeframe='3d',
    page_size=100,
)

print('status:', response.status_code)
for key, value in response.meta.items():
    print(f'{key}: {value}')

### The same pattern for every endpoint

Because every endpoint is queried the same way, you can drive them from a list of paths instead of repeating the same call. This tours the main data endpoints, printing the description from each response's `meta`:

In [ ]:
endpoints = [
    'research_outputs/attention',
    'research_outputs/demographics',
    'research_outputs/journals',
    'research_outputs/mentions',
    'research_outputs/mention_sources',
    'research_outputs',
]

for path in endpoints:
    response = api_client.get(path, page_size=1)
    print(f"{path} -> {response.meta.get('description')}")